In [ ]:
# ENHANCED VERSION - Milestone 2
# Improvements:
# - Added error handling for database connection
# - Refactored filtering logic into modular function
# - Improved query efficiency using MongoDB filtering
# - Added defensive programming for map updates

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dashboard components
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px

# Utilities
import base64
import pandas as pd

# Import CRUD operations
from crud import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU1234"

# Database connection with error handling
try:
    db = AnimalShelter(username, password)
    df = pd.DataFrame.from_records(db.read({}))
    
    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)
        
    print("Database connection successful")
    
except Exception as e:
    print("Database connection failed:", e)
    df = pd.DataFrame()

#########################
# Dashboard Layout / View
#########################

from dash import Dash
app = Dash(__name__)

# Load logo safely
try:
    image_filename = 'Grazioso_Salvare_Logo.png'
    encoded_image = base64.b64encode(open(image_filename, 'rb').read())
except Exception as e:
    print("Image load failed:", e)
    encoded_image = ""

app.layout = html.Div([
    html.Center(html.B(html.H1('CS-340 Dashboard (Enhanced)'))),
    
    html.Center(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()) if encoded_image else "",
            style={'height': '100px'}
        )
    ),

    html.Center(html.H4('Dashboard by Celeste Wally')),
    html.Hr(),

    # Dropdown filter
    html.Label('Select Rescue Type:'),
    dcc.Dropdown(
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'Water Rescue'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain or Wilderness Rescue'},
            {'label': 'Disaster or Individual Tracking', 'value': 'Disaster or Individual Tracking'},
            {'label': 'Reset', 'value': 'Reset'}
        ],
        value='Reset'
    ),

    html.Hr(),

    # Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left'}
    ),

    html.Br(),
    html.Hr(),

    # Graph and Map layout
    html.Div(style={'display': 'flex'}, children=[
        html.Div(id='graph-id', style={'width': '50%'}),
        html.Div(id='map-id', style={'width': '50%'})
    ])
])

#############################################
# HELPER FUNCTION (MODULAR DESIGN)
#############################################

def build_query(filter_type):
    queries = {
        'Water Rescue': {
            "breed": "Labrador Retriever Mix",
            "age_upon_outcome_in_weeks": {"$lte": 104}
        },
        'Mountain or Wilderness Rescue': {
            "breed": "German Shepherd",
            "age_upon_outcome_in_weeks": {"$lte": 104}
        },
        'Disaster or Individual Tracking': {
            "breed": "Belgian Malinois",
            "age_upon_outcome_in_weeks": {"$lte": 104}
        }
    }
    return queries.get(filter_type, {})

def process_data(records):
    try:
        df = pd.DataFrame.from_records(records)

        if 'breed' in df.columns:
            breed_counts = df['breed'].value_counts().to_dict()

        df_sorted = df.sort_values(by='age_upon_outcome_in_weeks', ascending=True)

        return df_sorted

    except Exception as e:
        print("Data processing error:", e)
        return pd.DataFrame()

#############################################
# CALLBACKS
#############################################

# Update table
@app.callback(Output('datatable-id', 'data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    try:
        query = build_query(filter_type)

        records = db.read(query)
        filtered_df = process_data(records)

        if '_id' in filtered_df.columns:
            filtered_df.drop(columns=['_id'], inplace=True)

        return filtered_df.to_dict('records')

    except Exception as e:
        print("Error updating dashboard:", e)
        return []

# Update graph
@app.callback(Output('graph-id', "children"),
              [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    try:
        if viewData is None or len(viewData) == 0:
            return []

        dff = pd.DataFrame.from_dict(viewData)
        fig = px.pie(dff, names='breed', title='Breed Distribution')

        return [dcc.Graph(figure=fig)]

    except Exception as e:
        print("Graph error:", e)
        return []


# Highlight columns
@app.callback(Output('datatable-id', 'style_data_conditional'),
              [Input('datatable-id', 'selected_columns')])
def update_styles(selected_columns):
    if selected_columns is None:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Update map with error handling
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    try:
        if viewData is None or index is None or len(index) == 0:
            return []

        dff = pd.DataFrame.from_dict(viewData)
        row = index[0]

        lat = dff.iloc[row, 13]
        lon = dff.iloc[row, 14]

        return [
            dl.Map(
                style={'width': '1000px', 'height': '500px'},
                center=[lat, lon],
                zoom=10,
                children=[
                    dl.TileLayer(),
                    dl.Marker(position=[lat, lon], children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ])
                ]
            )
        ]

    except Exception as e:
        print("Map error:", e)
        return []


# Run server
app.run(debug=True)

/Users/celestewally/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Read Error: nv-desktop-services.apporto.com:32117: [Errno 8] nodename nor servname provided, or not known (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 69c962a9ab233008d6324229, topology_type: Unknown, servers: [<ServerDescription ('nv-desktop-services.apporto.com', 32117) server_type: Unknown, rtt: None, error=AutoReconnect('nv-desktop-services.apporto.com:32117: [Errno 8] nodename nor servname provided, or not known (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>
Database connection successful


Read Error: nv-desktop-services.apporto.com:32117: [Errno 8] nodename nor servname provided, or not known (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 69c962a9ab233008d6324229, topology_type: Unknown, servers: [<ServerDescription ('nv-desktop-services.apporto.com', 32117) server_type: Unknown, rtt: None, error=AutoReconnect('nv-desktop-services.apporto.com:32117: [Errno 8] nodename nor servname provided, or not known (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>


In [ ]:
!python3 -m pip install jupyter-dash dash dash-leaflet plotly pandas pymongo

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
